In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install tensorflow tqdm


In [4]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from tqdm import tqdm  # for a progress bar

# Paths to your directories
images_dir = "/content/drive/MyDrive/Fetal Abdominal Structures Segmentation Dataset Using Ultrasonic Images (1)/IMAGES"
masks_dir = "/content/drive/MyDrive/Fetal Abdominal Structures Segmentation Dataset Using Ultrasonic Images (1)/IMAGES"

# Image and mask dimensions
IMG_HEIGHT = 224
IMG_WIDTH = 224

def load_images_and_masks(images_dir, masks_dir):
    # Check if directories exist
    if not os.path.exists(images_dir):
        raise FileNotFoundError(f"Image directory does not exist: {images_dir}")
    if not os.path.exists(masks_dir):
        raise FileNotFoundError(f"Mask directory does not exist: {masks_dir}")

    images = []
    masks = []

    # List all image and mask files
    image_files = sorted([f for f in os.listdir(images_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    mask_files = sorted([f for f in os.listdir(masks_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])

    # Debugging: Print the files found
    print(f"Found {len(image_files)} image files: {image_files}")
    print(f"Found {len(mask_files)} mask files: {mask_files}")

    if not image_files:
        raise FileNotFoundError(f"No valid image files found in directory: {images_dir}")
    if not mask_files:
        raise FileNotFoundError(f"No valid mask files found in directory: {masks_dir}")

    # Load images and masks with a progress bar
    for img_file, mask_file in tqdm(zip(image_files, mask_files), total=len(image_files), desc="Loading Images and Masks"):
        # Load the image
        img_path = os.path.join(images_dir, img_file)
        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        image = cv2.resize(image, (IMG_WIDTH, IMG_HEIGHT))
        image = image / 255.0  # Normalize pixel values
        images.append(image)

        # Load the corresponding mask
        mask_path = os.path.join(masks_dir, mask_file)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_WIDTH, IMG_HEIGHT))
        mask = mask / 255.0  # Normalize pixel values
        masks.append(mask)

    return np.array(images), np.array(masks)

# Load images and masks
try:
    X, Y = load_images_and_masks(images_dir, masks_dir)
except FileNotFoundError as e:
    print(e)
    X, Y = np.array([]), np.array([])  # Default fallback

if X.size > 0 and Y.size > 0:
    # Ensure masks are binary (0 or 1)
    Y = np.round(Y)  # Convert to binary (if it isn't already)

    # Expand dimensions for the masks (necessary for training)
    Y = np.expand_dims(Y, axis=-1)

    # Split dataset into training, validation, and testing
    X_train, X_temp, Y_train, Y_temp = train_test_split(X, Y, test_size=0.3, random_state=42)
    X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.5, random_state=42)

    print(f"Training Set: {X_train.shape}, {Y_train.shape}")
    print(f"Validation Set: {X_val.shape}, {Y_val.shape}")
    print(f"Test Set: {X_test.shape}, {Y_test.shape}")
else:
    print("Skipping dataset splitting due to missing or empty data.")



Found 1588 image files: ['P0100_IMG1.png', 'P0100_IMG2.png', 'P0100_IMG3.png', 'P0100_IMG4.png', 'P0100_IMG5.png', 'P0100_IMG6.png', 'P0100_IMG7.png', 'P0100_IMG8.png', 'P0101_IMG1.png', 'P0101_IMG10.png', 'P0101_IMG11.png', 'P0101_IMG12.png', 'P0101_IMG2.png', 'P0101_IMG3.png', 'P0101_IMG4.png', 'P0101_IMG5.png', 'P0101_IMG6.png', 'P0101_IMG7.png', 'P0101_IMG8.png', 'P0101_IMG9.png', 'P0102_IMG1.png', 'P0102_IMG2.png', 'P0102_IMG3.png', 'P0102_IMG4.png', 'P0102_IMG5.png', 'P0102_IMG6.png', 'P0102_IMG7.png', 'P0102_IMG8.png', 'P0102_IMG9.png', 'P0103_IMG1.png', 'P0103_IMG10.png', 'P0103_IMG11.png', 'P0103_IMG12.png', 'P0103_IMG13.png', 'P0103_IMG14.png', 'P0103_IMG15.png', 'P0103_IMG2.png', 'P0103_IMG3.png', 'P0103_IMG4.png', 'P0103_IMG5.png', 'P0103_IMG6.png', 'P0103_IMG7.png', 'P0103_IMG8.png', 'P0103_IMG9.png', 'P0104_IMG1.png', 'P0104_IMG10.png', 'P0104_IMG11.png', 'P0104_IMG12.png', 'P0104_IMG13.png', 'P0104_IMG14.png', 'P0104_IMG15.png', 'P0104_IMG2.png', 'P0104_IMG3.png', 'P0104

Loading Images and Masks: 100%|██████████| 1588/1588 [04:56<00:00,  5.36it/s]


Training Set: (1111, 224, 224, 3), (1111, 224, 224, 1)
Validation Set: (238, 224, 224, 3), (238, 224, 224, 1)
Test Set: (239, 224, 224, 3), (239, 224, 224, 1)


In [5]:
def geoproteonet(input_shape):
    # Input layer
    input_layer = layers.Input(shape=input_shape)

    # GeoProteoNet Core Layers (example)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_layer)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Bottleneck layers (may include skip connections)
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)

    # Decoder (Upsampling layers)
    x = layers.Conv2DTranspose(128, (2, 2), strides=2, padding='same')(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)

    x = layers.Conv2DTranspose(32, (2, 2), strides=2, padding='same')(x)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)

    # Output layer (Sigmoid for binary segmentation)
    output_layer = layers.Conv2D(1, (1, 1), activation='sigmoid', padding='same')(x)

    # Create Model
    model = tf.keras.models.Model(inputs=input_layer, outputs=output_layer)

    return model


In [ ]:
# Image Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

# Create an instance of the model by calling the geoproteonet function
# Assuming X_train has shape (num_samples, IMG_HEIGHT, IMG_WIDTH, 3)
input_shape = X_train.shape[1:]
model = geoproteonet(input_shape)

# Compile the model before training
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy']) # This line is added

# Train the model using augmented data
history = model.fit(
    train_datagen.flow(X_train, Y_train, batch_size=16),
    steps_per_epoch=len(X_train) // 16,
    epochs=20,
    validation_data=val_datagen.flow(X_val, Y_val, batch_size=16),
    validation_steps=len(X_val) // 16
)

Epoch 1/20
29/69 ━━━━━━━━━━━━━━━━━━━━ 4:41 7s/step - accuracy: 0.8961 - loss: 0.5644

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# Evaluate the model on test data
test_loss, test_accuracy = model.evaluate(X_test, Y_test)   # Assuming genomic_data_test is available
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

# Predict on the test set
predictions = model.predict(X_test)

# Threshold the predictions to get binary masks
predictions_binary = (predictions > 0.5).astype(np.uint8)

# Flatten arrays for easier evaluation metrics calculation
Y_test_flat = Y_test.flatten()
predictions_binary_flat = predictions_binary.flatten()

# Import necessary metrics functions
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score # Importing confusion_matrix, precision_score, recall_score, f1_score

# Calculate Confusion Matrix
cm = confusion_matrix(Y_test_flat, predictions_binary_flat)
print(f"Confusion Matrix: \n{cm}")

# Calculate Precision, Recall, F1 Score
precision = precision_score(Y_test_flat, predictions_binary_flat)
recall = recall_score(Y_test_flat, predictions_binary_flat)
f1 = f1_score(Y_test_flat, predictions_binary_flat)

# Calculate Positive Sensitivity Value (PSV) and Recall Index (RI)
TP = np.sum((Y_test == 1) & (predictions_binary == 1))  # True Positive
FN = np.sum((Y_test == 1) & (predictions_binary == 0))  # False Negative
PSV = TP / (TP + FN)  # Positive Sensitivity Value

RI = TP / (TP + FN)  # Recall Index (same as recall)

# Calculate SD Ratio (Standard Deviation Ratio)
SD_pred = np.std(predictions_binary)
SD_gt = np.std(Y_test)
SD_ratio = SD_pred / SD_gt

# Calculate Dice Coefficient
intersection = np.sum((Y_test_flat == 1) & (predictions_binary_flat == 1))
union = np.sum((Y_test_flat == 1) | (predictions_binary_flat == 1))
dice_coefficient = 2 * intersection / (union + intersection)

# Print all metrics
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
print(f"PSV (Positive Sensitivity Value): {PSV}")
print(f"RI (Recall Index): {RI}")
print(f"SD Ratio (Standard Deviation Ratio): {SD_ratio}")
print(f"Dice Coefficient: {dice_coefficient}")


In [ ]:
# @title Save Weights for Backend
# Run this cell after your model has finished training to save the weights.
# You can then copy the generated 'geoproteonet.weights.h5' file into the 'backend' folder of your project.
model.save_weights("geoproteonet.weights.h5")
print("Weights successfully saved as 'geoproteonet.weights.h5'!")
print("Copy this file into your project's 'backend' folder to use the trained AI.")
